# Build mechanism result tables

This orchestrator builds three separate outputs: Volt-VAr net-meter-proxy results, Volt-Watt net-meter-proxy results, and response observability. It never combines them into one score.

Counterfactual-supported curtailment is intentionally not built: load-PV decomposition and its uncertainty validation have not passed methodology gate 7.


In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import duckdb
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

HERE = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'src' / 'dnsp_analysis').is_dir()), None)
assert PROJECT_ROOT is not None, 'Start Jupyter inside the dnsp_analysis project.'
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from dnsp_analysis.as4777_curves import (
    VOLT_VAR, VOLT_WATT, vvar_required_q, vvar_required_q_sql,
    vw_max_p, vw_max_p_sql,
)
from dnsp_analysis.config import load_config
from dnsp_analysis.db import connect
from dnsp_analysis.mechanism_config import load_mechanism_config
from dnsp_analysis.mechanism_paths import (
    response_observability_path, sign_candidate_days_path,
    sign_phase_intervals_path, sign_site_intervals_path,
    voltvar_results_path, voltwatt_results_path,
)
from dnsp_analysis.mechanism_results import (
    build_response_observability, build_voltvar_results, build_voltwatt_results,
)
from dnsp_analysis.mechanism_validation import validate_mechanism_results
from dnsp_analysis.schemas import sql_string
from dnsp_analysis.sign_diagnostics import build_sign_diagnostics

CONFIG_PATH = PROJECT_ROOT / 'analysis.toml'
config = load_config(CONFIG_PATH, check_inputs=True)
mechanism = load_mechanism_config(CONFIG_PATH)
pd.set_option('display.max_columns', 100)
plt.style.use('seaborn-v0_8-whitegrid')


## Stage 0: methodology readiness

The configured voltage is the maximum across inferred DER phases. This is a conservative high-voltage exposure choice, not a claim about the inverter's internal control logic. Per-phase and min/mean/max voltage remain in the structured source. The only permitted capacity basis is verified `s_rated_kva`; null ratings remain not assessable.


In [ ]:
methodology = pd.Series({
    'methodology_id': mechanism.methodology_id,
    'voltage_aggregate': mechanism.voltage_aggregate,
    'voltage_basis': mechanism.voltage_basis_label,
    'capacity_basis': mechanism.capacity_basis,
    'active_sign_review_state': mechanism.active_sign_review_state,
    'reactive_sign_review_state': mechanism.reactive_sign_review_state,
    'measurement_basis': 'net_meter_proxy',
    'voltage_location': 'revenue_meter',
    'curtailment': 'not_built_gate_7_unmet',
}, name='value')
display(methodology.to_frame())
assert mechanism.capacity_basis == 's_rated_kva'
assert mechanism.voltage_aggregate in {'avg', 'max'}


## Stage 1: empirical P/Q sign review

This stage selects 2–3 deterministic solar-only, no-battery, no-controlled-load site-days from the configured month. Selection uses high-voltage exposure and positive derived net export, not the desired response direction. Inspect raw and derived P/Q. At high voltage, generator-convention Q should move negative for absorption. If either hypothesis is contradicted, stop, correct the foundation sign configuration, and rebuild affected canonical/structured outputs before continuing. DNSP confirmation is still required.


In [ ]:
OVERWRITE_SIGN_DIAGNOSTICS = True
sign_paths_exist = all(p.is_file() for p in (
    sign_candidate_days_path(config), sign_site_intervals_path(config),
    sign_phase_intervals_path(config),
))
if OVERWRITE_SIGN_DIAGNOSTICS or not sign_paths_exist:
    sign_summary = build_sign_diagnostics(config, mechanism, overwrite=OVERWRITE_SIGN_DIAGNOSTICS)
    display(pd.DataFrame([sign_summary]).T.rename(columns={0: 'value'}))
else:
    print('Reusing sign diagnostics. Set OVERWRITE_SIGN_DIAGNOSTICS=True to rebuild.')
con = connect(config)
candidate_days = con.execute(f'SELECT * FROM read_parquet({sql_string(sign_candidate_days_path(config))})').fetchdf()
sign_site = con.execute(f'SELECT * FROM read_parquet({sql_string(sign_site_intervals_path(config))}) ORDER BY serial, timestamp_utc').fetchdf()
sign_phase = con.execute(f'SELECT * FROM read_parquet({sql_string(sign_phase_intervals_path(config))}) ORDER BY serial, timestamp_utc, phase').fetchdf()
con.close()
display(candidate_days)
display(sign_phase[['serial','timestamp_local','phase','voltage_v','active_power_raw_w','p_export_w','reactive_power_raw_var','q_absorbing_var','q_generator_var','is_inferred_der_phase']].head(30))
assert candidate_days.serial.nunique() == mechanism.sign_audit_site_count
assert sign_site.timestamp_utc.notna().all()


In [ ]:
for serial, frame in sign_site.groupby('serial', sort=True):
    daylight = frame.loc[frame.local_hour.between(8, 17)].copy()
    fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
    axes[0].plot(daylight.timestamp_local, daylight.comparison_voltage_v, color='tab:orange')
    axes[0].axhspan(253, 258, alpha=0.12, color='red')
    axes[0].set_ylabel('Revenue-meter V')
    axes[1].plot(daylight.timestamp_local, daylight.p_export_net_proxy_w / 1000, color='tab:green')
    axes[1].axhline(0, color='black', linewidth=0.7)
    axes[1].set_ylabel('Net export proxy kW')
    axes[2].plot(daylight.timestamp_local, daylight.q_generator_net_proxy_var / 1000, label='Q generator (+ supply / - absorb)')
    axes[2].plot(daylight.timestamp_local, daylight.q_absorbing_net_proxy_var / 1000, alpha=0.45, label='Q absorbing (+ absorb)')
    axes[2].axhline(0, color='black', linewidth=0.7)
    axes[2].set_ylabel('Net Q proxy kvar')
    axes[2].legend()
    fig.suptitle(f'Sign diagnostic — site {serial}')
    plt.tight_layout()
    plt.show()


In [ ]:
SIGN_REVIEW_CONFIRMATION = 'SIGN REVIEW COMPLETE'
# Review the candidate plots and raw/derived sign table above, then choose
# exactly one of the two documented confirmation strings:
#
#   'SIGN REVIEW COMPLETE'
#       Both active_sign_review_state and reactive_sign_review_state in
#       [mechanism_analysis] must already be a ready state
#       (empirically_supported_pending_provider_confirmation or
#       provider_confirmed). Unlocks assessable magnitude/observability rows.
#
#   'BUILD WITH SIGNS UNVERIFIED'
#       Proceeds with both review states left exactly as configured
#       (currently 'unverified' -- DNSP has not confirmed either
#       convention). This does NOT flip, fake, or bypass the sign states: it
#       is a documented, deliberate choice to build the denominator/coverage
#       and observability tables now, with every sign-dependent row labelled
#       'sign_unverified' (mechanism_results.py already implements this --
#       it is the same behaviour a ready-sign build would fall back to for
#       any row failing the sign gate, just applied to every row). No
#       'assessable' proxy curve-status or observability-direction row is
#       produced under this path. Flip the two review states in
#       analysis.toml once DNSP confirms (or an empirical review supports
#       a state), then rerun the Stage 3-5 builders with
#       OVERWRITE_SAMPLE/OVERWRITE_FULL=True to rebuild a clean, assessable
#       database. The methodology_id and both review states are stamped on
#       every output row, so a sign_unverified-built table can never be
#       confused with a ready-sign build later.
assert SIGN_REVIEW_CONFIRMATION in {'SIGN REVIEW COMPLETE', 'BUILD WITH SIGNS UNVERIFIED'}, (
    'Review the candidate plots and raw/derived sign table first, then set '
    "SIGN_REVIEW_CONFIRMATION to one of the two documented strings."
)
if SIGN_REVIEW_CONFIRMATION == 'SIGN REVIEW COMPLETE':
    assert mechanism.active_sign_review_state not in {'contradicted', 'inconclusive', 'unverified'}
    assert mechanism.reactive_sign_review_state not in {'contradicted', 'inconclusive', 'unverified'}
    assert mechanism.active_sign_ready and mechanism.reactive_sign_ready, (
        'Record the reviewed states in [mechanism_analysis], restart the kernel, and rerun. '
        'If contradicted, fix analysis.toml signs and rebuild the foundation/structured data first.'
    )
    print('Sign review gate passed; provider confirmation may still be pending.')
else:
    assert mechanism.active_sign_review_state == 'unverified'
    assert mechanism.reactive_sign_review_state == 'unverified'
    assert not (mechanism.active_sign_ready and mechanism.reactive_sign_ready)
    print(
        "Sign review gate passed under 'BUILD WITH SIGNS UNVERIFIED'. Both review states remain "
        "'unverified'. Every magnitude/observability row that depends on either sign will be "
        "labelled sign_unverified -- not assessable, not conforming. See Stage 1 markdown and "
        "docs/MECHANISM_RESULTS.md for what this does and does not establish."
    )


## Stage 2: curve and SQL parity

The curves use generator convention directly: `Q4=-0.60`. No curve sign flip is applied.


In [ ]:
voltages = [200.0, 207.0, 213.5, 220.0, 240.0, 249.0, 253.0, 258.0, 260.0, 265.0]
curve_rows = []
for voltage in voltages:
    sql_vv, sql_vw = duckdb.sql(
        'SELECT ' + vvar_required_q_sql(str(voltage), '1.0') + ', ' + vw_max_p_sql(str(voltage), '1.0')
    ).fetchone()
    curve_rows.append({
        'voltage_v': voltage,
        'q_fraction_python': vvar_required_q(voltage, 1.0),
        'q_fraction_sql': float(sql_vv),
        'p_fraction_python': vw_max_p(voltage, 1.0),
        'p_fraction_sql': float(sql_vw),
    })
curve_check = pd.DataFrame(curve_rows)
display(curve_check)
assert (curve_check.q_fraction_python - curve_check.q_fraction_sql).abs().max() < 1e-12
assert (curve_check.p_fraction_python - curve_check.p_fraction_sql).abs().max() < 1e-12
assert vvar_required_q(VOLT_VAR.v4, 1.0) == -0.60


## Stage 3: deterministic-slice Volt-VAr proxy table


In [ ]:
SAMPLE_MONTH = '2025-04'
SAMPLE_SITE_BUCKET = 0
OVERWRITE_SAMPLE = True
sample_scope = config.scope(SAMPLE_MONTH, SAMPLE_SITE_BUCKET)
if OVERWRITE_SAMPLE or not voltvar_results_path(config, sample_scope).is_file():
    display(pd.DataFrame([build_voltvar_results(config, sample_scope, mechanism, overwrite=OVERWRITE_SAMPLE)]).T.rename(columns={0: 'value'}))
con = connect(config)
vv_sample = con.execute(f'SELECT * FROM read_parquet({sql_string(voltvar_results_path(config, sample_scope))}) ORDER BY serial, year_utc, month_utc, voltage_bin_lower_v LIMIT 100').fetchdf()
con.close()
display(vv_sample)
assert not vv_sample.formal_inverter_conformance_assessable.any()
assert vv_sample.measurement_basis.eq('net_meter_proxy').all()


## Stage 4: deterministic-slice Volt-Watt proxy table


In [ ]:
if OVERWRITE_SAMPLE or not voltwatt_results_path(config, sample_scope).is_file():
    display(pd.DataFrame([build_voltwatt_results(config, sample_scope, mechanism, overwrite=OVERWRITE_SAMPLE)]).T.rename(columns={0: 'value'}))
con = connect(config)
vw_sample = con.execute(f'SELECT * FROM read_parquet({sql_string(voltwatt_results_path(config, sample_scope))}) ORDER BY serial, year_utc, month_utc, voltage_bin_lower_v LIMIT 100').fetchdf()
con.close()
display(vw_sample)
assert vw_sample.interpretation_guardrail.str.contains('not proof').all()
assert not vw_sample.formal_inverter_conformance_assessable.any()


## Stage 5: deterministic-slice response observability

Observability asks whether a directional response is visible in net-meter P/Q. It is not a conformance score.


In [ ]:
if OVERWRITE_SAMPLE or not response_observability_path(config, sample_scope).is_file():
    display(pd.DataFrame([build_response_observability(config, sample_scope, mechanism, overwrite=OVERWRITE_SAMPLE)]).T.rename(columns={0: 'value'}))
con = connect(config)
response_sample = con.execute(f'SELECT * FROM read_parquet({sql_string(response_observability_path(config, sample_scope))}) ORDER BY serial, year_utc, month_utc, phase').fetchdf()
con.close()
display(response_sample.groupby(['voltvar_observability_status','voltwatt_observability_status'], dropna=False).size().rename('n_site_phase_months').reset_index())
assert response_sample.observability_only.all()
assert not response_sample.formal_inverter_conformance_assessable.any()


## Stage 6: deterministic-slice denominator, key, coverage and provenance checks


In [ ]:
sample_validation = validate_mechanism_results(config, sample_scope)
headline = {k: v for k, v in sample_validation.items() if k not in {'failures', 'monthly_coverage'}}
display(pd.DataFrame([headline]).T.rename(columns={0: 'value'}))
display(pd.DataFrame(sample_validation['monthly_coverage']))
reason_counts = pd.Series({
    'VV assessable': sample_validation['voltvar_assessable_intervals'],
    'VV missing verified capacity': sample_validation['voltvar_capacity_unavailable_intervals'],
    'VW assessable': sample_validation['voltwatt_assessable_intervals'],
    'VW missing verified capacity': sample_validation['voltwatt_capacity_unavailable_intervals'],
})
fig, ax = plt.subplots(figsize=(8, 3.5))
reason_counts.plot.bar(ax=ax, color=['#4472c4','#a5a5a5','#70ad47','#a5a5a5'])
ax.set_ylabel('Intervals')
ax.set_title('Magnitude-assessment coverage (separate mechanisms)')
plt.tight_layout(); plt.show()
assert sample_validation['status'] == 'pass'
assert sample_validation['voltvar_denominator_difference'] == 0
assert sample_validation['voltwatt_denominator_difference'] == 0
assert sample_validation['voltvar_duplicate_keys'] == 0
assert sample_validation['voltwatt_duplicate_keys'] == 0
assert sample_validation['response_duplicate_keys'] == 0


## Optional full-dataset build

Run each cell separately. The full build stays locked until the sign review is recorded in configuration and the deterministic slice passes. With current null `s_rated_kva`, Volt-VAr and Volt-Watt magnitude rows will honestly remain not assessable; response observability can still be populated.


In [ ]:
FULL_RUN_CONFIRMATION = 'RUN MECHANISM RESULTS FULL'  # Change to: RUN MECHANISM RESULTS FULL
OVERWRITE_FULL = True
assert sample_validation['status'] == 'pass'
assert SIGN_REVIEW_CONFIRMATION in {'SIGN REVIEW COMPLETE', 'BUILD WITH SIGNS UNVERIFIED'}, (
    'Rerun the Stage 1 sign-review gate cell in this session first.'
)
if SIGN_REVIEW_CONFIRMATION == 'SIGN REVIEW COMPLETE':
    assert mechanism.active_sign_ready and mechanism.reactive_sign_ready
else:
    print(
        "Full build proceeding under 'BUILD WITH SIGNS UNVERIFIED': every sign-dependent row in "
        "the full result tables will be sign_unverified, not assessable."
    )
assert FULL_RUN_CONFIRMATION == 'RUN MECHANISM RESULTS FULL'
full_scope = config.scope(None, None)
print('Full mechanism build unlocked:', full_scope.label)


In [ ]:
if OVERWRITE_FULL or not voltvar_results_path(config, full_scope).is_file():
    full_vv = build_voltvar_results(config, full_scope, mechanism, overwrite=OVERWRITE_FULL)
    display(pd.DataFrame([full_vv]).T.rename(columns={0: 'value'}))
else:
    print('Reusing full Volt-VAr proxy results.')


In [ ]:
if OVERWRITE_FULL or not voltwatt_results_path(config, full_scope).is_file():
    full_vw = build_voltwatt_results(config, full_scope, mechanism, overwrite=OVERWRITE_FULL)
    display(pd.DataFrame([full_vw]).T.rename(columns={0: 'value'}))
else:
    print('Reusing full Volt-Watt proxy results.')


In [ ]:
if OVERWRITE_FULL or not response_observability_path(config, full_scope).is_file():
    full_response = build_response_observability(config, full_scope, mechanism, overwrite=OVERWRITE_FULL)
    display(pd.DataFrame([full_response]).T.rename(columns={0: 'value'}))
else:
    print('Reusing full response-observability results.')


In [ ]:
full_validation = validate_mechanism_results(config, full_scope)
display(pd.DataFrame([{k: v for k, v in full_validation.items() if k not in {'failures','monthly_coverage'}}]).T.rename(columns={0: 'value'}))
display(pd.DataFrame(full_validation['monthly_coverage']))
assert full_validation['status'] == 'pass'
assert full_validation['counterfactual_supported_curtailment'] == 'not_built_gate_7_unmet'
print('Full mechanism result build passed structural and methodological validation.')


## Phase-scope comparison: all_phases vs der_inferred (optional)

Everything above uses `phase_scope_basis = "der_inferred"` (the default):
comparison voltage, and the summed P/Q, come only from the phase(s)
`site_eligibility.inferred_der_phases` marks as DER-connected for that site
-- which can be 1 of 3 phases.

For a DER genuinely wired to a single phase, that phase's own voltage *is*
its terminal voltage as far as the standard's Volt-Watt/Volt-Var curves are
concerned -- there is no real ambiguity there, since the inverter has no
electrical connection to the other two phases. The genuine open questions
are (a) how a truly multi-phase inverter internally derives one control
voltage from three unbalanced phases -- AS/NZS 4777.2:2020 does not
prescribe this -- and (b) whether `site_eligibility`'s phase-mapping
inference is actually correct for a given site (`phase_mapping_confidence`
is a statistical inference from load signatures, not a confirmed fact).

`phase_scope_basis = "all_phases"` is a deliberate sensitivity/comparison
run, not a claim that the standard requires averaging in phases the DER may
not be connected to: comparison voltage becomes `voltage_mean_valid_v` /
`voltage_max_valid_v` (every phase), and P/Q become `p_export_net_observed_w`
/ `q_absorbing_net_observed_var` (summed over every phase with a power
reading, gated by a recomputed all-phase completeness check so a partial-
phase sum is never silently treated as a verified total). Summing P/Q is
correct under either scope -- that part is just energy accounting, not a
modeling choice; only the voltage aggregation is genuinely a choice.

This section writes to a separate `phase_scope_all_phases/` subfolder --
it never overwrites the der_inferred results already built above, so both
can be inspected side by side. `response_observability.parquet` is not
rebuilt here: it already reports every actual telemetry phase (tagged
`is_inferred_der_phase`), so it does not depend on this setting.

In [ ]:
import dataclasses

mechanism_all_phases = dataclasses.replace(
    mechanism, phase_scope_basis='all_phases'
).validate()
display(pd.Series({
    'methodology_id': mechanism_all_phases.methodology_id,
    'phase_scope_basis': mechanism_all_phases.phase_scope_basis,
    'voltage_basis': mechanism_all_phases.voltage_basis_label,
    'comparison_p_column': mechanism_all_phases.comparison_p_column,
    'comparison_q_absorbing_column': mechanism_all_phases.comparison_q_absorbing_column,
}, name='value').to_frame())
print('Original der_inferred methodology_id (unchanged, still the paths used above):',
      mechanism.methodology_id)

In [ ]:
OVERWRITE_SAMPLE_ALL_PHASES = True
if OVERWRITE_SAMPLE_ALL_PHASES or not voltvar_results_path(config, sample_scope, mechanism_all_phases).is_file():
    display(pd.DataFrame([build_voltvar_results(
        config, sample_scope, mechanism_all_phases, overwrite=OVERWRITE_SAMPLE_ALL_PHASES
    )]).T.rename(columns={0: 'value'}))
if OVERWRITE_SAMPLE_ALL_PHASES or not voltwatt_results_path(config, sample_scope, mechanism_all_phases).is_file():
    display(pd.DataFrame([build_voltwatt_results(
        config, sample_scope, mechanism_all_phases, overwrite=OVERWRITE_SAMPLE_ALL_PHASES
    )]).T.rename(columns={0: 'value'}))

sample_validation_all_phases = validate_mechanism_results(config, sample_scope, mechanism_all_phases)
assert sample_validation_all_phases['status'] == 'pass'
assert sample_validation_all_phases['voltvar_denominator_difference'] == 0
assert sample_validation_all_phases['voltwatt_denominator_difference'] == 0
print('Sample all_phases build validated.')

### Full all_phases build (deliberate opt-in, mirrors the main gate above)

In [ ]:
FULL_RUN_CONFIRMATION_ALL_PHASES = 'RUN MECHANISM RESULTS FULL ALL PHASES'  # Change to: RUN MECHANISM RESULTS FULL ALL PHASES
OVERWRITE_FULL_ALL_PHASES = True
assert sample_validation_all_phases['status'] == 'pass'
assert SIGN_REVIEW_CONFIRMATION in {'SIGN REVIEW COMPLETE', 'BUILD WITH SIGNS UNVERIFIED'}
assert FULL_RUN_CONFIRMATION_ALL_PHASES == 'RUN MECHANISM RESULTS FULL ALL PHASES'
print('Full all_phases mechanism build unlocked:', full_scope.label)

In [ ]:
if OVERWRITE_FULL_ALL_PHASES or not voltvar_results_path(config, full_scope, mechanism_all_phases).is_file():
    full_vv_ap = build_voltvar_results(config, full_scope, mechanism_all_phases, overwrite=OVERWRITE_FULL_ALL_PHASES)
    display(pd.DataFrame([full_vv_ap]).T.rename(columns={0: 'value'}))
else:
    print('Reusing full all_phases Volt-VAr proxy results.')

if OVERWRITE_FULL_ALL_PHASES or not voltwatt_results_path(config, full_scope, mechanism_all_phases).is_file():
    full_vw_ap = build_voltwatt_results(config, full_scope, mechanism_all_phases, overwrite=OVERWRITE_FULL_ALL_PHASES)
    display(pd.DataFrame([full_vw_ap]).T.rename(columns={0: 'value'}))
else:
    print('Reusing full all_phases Volt-Watt proxy results.')

full_validation_all_phases = validate_mechanism_results(config, full_scope, mechanism_all_phases)
assert full_validation_all_phases['status'] == 'pass'
print('Full all_phases mechanism build passed structural validation.')

### Side-by-side: der_inferred vs all_phases

Both tables now exist at their own paths. This compares source-interval
counts and top-level denominator/classification totals -- it does not
merge them into a single score (`phase_scope` differs in meaning between
the two, so joining row-for-row on it would be misleading).

In [ ]:
def _totals(path, label):
    df = con.execute(f"SELECT * FROM read_parquet({sql_string(str(path))})").fetchdf()
    numeric = df.select_dtypes('number').sum(numeric_only=True)
    numeric['phase_scope_basis'] = label
    return numeric

con = connect(config)
der_totals = _totals(voltvar_results_path(config, full_scope, mechanism), 'der_inferred')
all_totals = _totals(voltvar_results_path(config, full_scope, mechanism_all_phases), 'all_phases')
con.close()

compare = pd.DataFrame([der_totals, all_totals]).set_index('phase_scope_basis')
display(compare[[c for c in compare.columns if c.startswith('n_')]].T)
print('\nA large gap here is a direct measure of how much the DER-phase mapping '
      'assumption is driving the Volt-VAr result -- not evidence either scope is "correct".')

## Capacity proxy comparison (optional): bracketing S_rated

`s_rated_kva` remains null for this fleet (no verified inverter rating source
exists yet), so every result above under the default `capacity_basis` stays
honestly `capacity_unavailable`/not assessable -- that has not changed and
this section does not change it.

This section adds two separate, explicitly named, separately-decided proxy
tracks (2026-08-03), each writing to its own namespaced path, never touching
the `s_rated_kva` results above:

- `solar_capacity_kw_proxy` -- DC solar nameplate from metadata. Known to
  **overstate** true inverter S_rated (residential DC:AC oversizing is
  common), which widens every tolerance band/ceiling and biases the
  assessment **lenient**.
- `p99_net_export_proxy` -- the 99th percentile of each site's own observed
  net-export power (generation minus house load), computed empirically by
  `capacity_proxy.py` from the full `structured_site_intervals` history.
  Known to **understate** true inverter S_rated (net export is always <=
  gross generation), which biases the assessment **conservative**.

Building both brackets the true (unknown) S_rated between a lenient and a
conservative estimate. Neither is treated as verified, and
`formal_inverter_conformance_assessable` stays `False` under both -- see
`mechanism_config.py`'s `CAPACITY_BASES` comment for the full rationale.

In [ ]:
import dataclasses

mechanism_solar_proxy = dataclasses.replace(
    mechanism, capacity_basis='solar_capacity_kw_proxy'
).validate()
mechanism_p99_proxy = dataclasses.replace(
    mechanism, capacity_basis='p99_net_export_proxy'
).validate()

display(pd.DataFrame([
    {'capacity_basis': m.capacity_basis, 'methodology_id': m.methodology_id,
     'label': m.capacity_basis_label}
    for m in (mechanism, mechanism_solar_proxy, mechanism_p99_proxy)
]))

### Step 1 -- build the empirical P99 capacity-proxy table (deterministic slice)

`capacity_proxy.py` computes this once per site from `p_export_der_phase_net_complete_w`
(or the `all_phases` equivalent), gated to the same core eligibility gate the
mechanism builders use. It is not a mechanism result -- it lives under
`analysis_cohort/` alongside `site_eligibility.parquet`, which it is joined
against, and is only consumed when `capacity_basis='p99_net_export_proxy'`.

In [ ]:
from dnsp_analysis.capacity_proxy import build_capacity_proxy
from dnsp_analysis.mechanism_paths import capacity_proxy_path

OVERWRITE_CAPACITY_PROXY_SAMPLE = True
if OVERWRITE_CAPACITY_PROXY_SAMPLE or not capacity_proxy_path(config, sample_scope, mechanism_p99_proxy).is_file():
    proxy_sample_summary = build_capacity_proxy(config, sample_scope, mechanism_p99_proxy, overwrite=OVERWRITE_CAPACITY_PROXY_SAMPLE)
    display(pd.DataFrame([proxy_sample_summary]).T.rename(columns={0: 'value'}))
else:
    print('Reusing sample capacity-proxy table.')

### Step 2 -- deterministic-slice builds under both proxies

In [ ]:
OVERWRITE_SAMPLE_CAPACITY_PROXY = True
for label, proxy_mechanism in (('solar_capacity_kw_proxy', mechanism_solar_proxy), ('p99_net_export_proxy', mechanism_p99_proxy)):
    if OVERWRITE_SAMPLE_CAPACITY_PROXY or not voltvar_results_path(config, sample_scope, proxy_mechanism).is_file():
        vv = build_voltvar_results(config, sample_scope, proxy_mechanism, overwrite=OVERWRITE_SAMPLE_CAPACITY_PROXY)
        display(pd.DataFrame([vv]).T.rename(columns={0: 'value'}))
    if OVERWRITE_SAMPLE_CAPACITY_PROXY or not voltwatt_results_path(config, sample_scope, proxy_mechanism).is_file():
        vw = build_voltwatt_results(config, sample_scope, proxy_mechanism, overwrite=OVERWRITE_SAMPLE_CAPACITY_PROXY)
        display(pd.DataFrame([vw]).T.rename(columns={0: 'value'}))
    validation = validate_mechanism_results(config, sample_scope, proxy_mechanism)
    assert validation['status'] == 'pass', f'{label}: sample validation failed -- {validation["failures"]}'
    print(f"{label}: sample n_assessable (Volt-VAr) = {validation['voltvar_assessable_intervals']:,}, "
          f"(Volt-Watt) = {validation['voltwatt_assessable_intervals']:,}")

### Full-dataset capacity-proxy build (deliberate opt-in)

For an apples-to-apples full-dataset comparison against the `s_rated_kva`
track above, that track should also be rebuilt under the current sign
review state (`OVERWRITE_FULL = True` in the Stage-labelled cells above) --
otherwise it is still comparing against a stale, sign-unverified-only build.

In [ ]:
FULL_RUN_CONFIRMATION_CAPACITY_PROXY = 'RUN MECHANISM RESULTS FULL CAPACITY PROXY'  # Change to: RUN MECHANISM RESULTS FULL CAPACITY PROXY
OVERWRITE_FULL_CAPACITY_PROXY = True
assert SIGN_REVIEW_CONFIRMATION in {'SIGN REVIEW COMPLETE', 'BUILD WITH SIGNS UNVERIFIED'}
assert FULL_RUN_CONFIRMATION_CAPACITY_PROXY == 'RUN MECHANISM RESULTS FULL CAPACITY PROXY'
print('Full capacity-proxy build unlocked:', full_scope.label)

In [ ]:
if OVERWRITE_FULL_CAPACITY_PROXY or not capacity_proxy_path(config, full_scope, mechanism_p99_proxy).is_file():
    full_proxy_summary = build_capacity_proxy(config, full_scope, mechanism_p99_proxy, overwrite=OVERWRITE_FULL_CAPACITY_PROXY)
    display(pd.DataFrame([full_proxy_summary]).T.rename(columns={0: 'value'}))
else:
    print('Reusing full capacity-proxy table.')

for label, proxy_mechanism in (('solar_capacity_kw_proxy', mechanism_solar_proxy), ('p99_net_export_proxy', mechanism_p99_proxy)):
    if OVERWRITE_FULL_CAPACITY_PROXY or not voltvar_results_path(config, full_scope, proxy_mechanism).is_file():
        vv = build_voltvar_results(config, full_scope, proxy_mechanism, overwrite=OVERWRITE_FULL_CAPACITY_PROXY)
        display(pd.DataFrame([vv]).T.rename(columns={0: 'value'}))
    else:
        print(f'Reusing full {label} Volt-VAr proxy results.')
    if OVERWRITE_FULL_CAPACITY_PROXY or not voltwatt_results_path(config, full_scope, proxy_mechanism).is_file():
        vw = build_voltwatt_results(config, full_scope, proxy_mechanism, overwrite=OVERWRITE_FULL_CAPACITY_PROXY)
        display(pd.DataFrame([vw]).T.rename(columns={0: 'value'}))
    else:
        print(f'Reusing full {label} Volt-Watt proxy results.')
    full_proxy_validation = validate_mechanism_results(config, full_scope, proxy_mechanism)
    assert full_proxy_validation['status'] == 'pass'
    print(f"{label}: FULL n_assessable (Volt-VAr) = {full_proxy_validation['voltvar_assessable_intervals']:,}, "
          f"(Volt-Watt) = {full_proxy_validation['voltwatt_assessable_intervals']:,}")

### Three-way bracket: `s_rated_kva` vs `solar_capacity_kw_proxy` vs `p99_net_export_proxy`

`s_rated_kva` is read from whatever is already built at its default path above
-- rerun that section with `OVERWRITE_FULL = True` first if it still predates
the current sign review state, or this row will show a stale comparison.

In [ ]:
def _capacity_totals(mechanism_variant, label):
    df = con.execute(
        f"SELECT * FROM read_parquet('{voltvar_results_path(config, full_scope, mechanism_variant)}')"
    ).fetchdf()
    numeric = df.select_dtypes('number').sum(numeric_only=True)
    numeric['capacity_basis'] = label
    return numeric

con = connect(config)
capacity_compare = pd.DataFrame([
    _capacity_totals(mechanism, 's_rated_kva (default, unverified)'),
    _capacity_totals(mechanism_solar_proxy, 'solar_capacity_kw_proxy (lenient bound)'),
    _capacity_totals(mechanism_p99_proxy, 'p99_net_export_proxy (conservative bound)'),
]).set_index('capacity_basis')
con.close()

display(capacity_compare[[c for c in capacity_compare.columns if c.startswith('n_')]].T)
print(
    '\nThe gap between the two proxy rows brackets how much the capacity-basis '
    'choice alone is driving the result -- not evidence either bound is correct. '
    'The s_rated_kva row should stay at 0 assessable everywhere; if it does not, '
    'a verified rating source has appeared and this whole section may no longer '
    'be necessary.'
)